#🏏 **CRICKET SCORE PREDICTOR USING ML 🏏**

---


# Import Libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

#Load the Datasets

In [3]:
balls = pd.read_csv("IPL_Ball_by_Ball_2008_2022.csv")
matches = pd.read_csv("IPL_Matches_2008_2022.csv")

# Data Information

###Match Dataset

In [4]:
print(matches.head())
print("\n------------------------------------------\n")
print(matches.shape)
print("\n------------------------------------------\n")
print(matches.columns)

        ID       City        Date Season  MatchNumber  \
0  1312200  Ahmedabad  2022-05-29   2022        Final   
1  1312199  Ahmedabad  2022-05-27   2022  Qualifier 2   
2  1312198    Kolkata  2022-05-25   2022   Eliminator   
3  1312197    Kolkata  2022-05-24   2022  Qualifier 1   
4  1304116     Mumbai  2022-05-22   2022           70   

                         Team1                 Team2  \
0             Rajasthan Royals        Gujarat Titans   
1  Royal Challengers Bangalore      Rajasthan Royals   
2  Royal Challengers Bangalore  Lucknow Super Giants   
3             Rajasthan Royals        Gujarat Titans   
4          Sunrisers Hyderabad          Punjab Kings   

                              Venue            TossWinner TossDecision  \
0  Narendra Modi Stadium, Ahmedabad      Rajasthan Royals          bat   
1  Narendra Modi Stadium, Ahmedabad      Rajasthan Royals        field   
2             Eden Gardens, Kolkata  Lucknow Super Giants        field   
3             Eden Garde

###Ball Dataset

In [5]:
print(balls.head())
print("\n------------------------------------------\n")
print(balls.shape)
print("\n------------------------------------------\n")
print(balls.columns)

        ID  innings  overs  ballnumber       batter          bowler  \
0  1312200        1      0           1  YBK Jaiswal  Mohammed Shami   
1  1312200        1      0           2  YBK Jaiswal  Mohammed Shami   
2  1312200        1      0           3   JC Buttler  Mohammed Shami   
3  1312200        1      0           4  YBK Jaiswal  Mohammed Shami   
4  1312200        1      0           5  YBK Jaiswal  Mohammed Shami   

   non-striker extra_type  batsman_run  extras_run  total_run  non_boundary  \
0   JC Buttler        NaN          0.0         0.0        0.0           0.0   
1   JC Buttler    legbyes          0.0         1.0        1.0           0.0   
2  YBK Jaiswal        NaN          1.0         0.0        1.0           0.0   
3   JC Buttler        NaN          0.0         0.0        0.0           0.0   
4   JC Buttler        NaN          0.0         0.0        0.0           0.0   

   isWicketDelivery player_out kind fielders_involved       BattingTeam  
0               0.0     

#Check Missing Values

In [6]:
print(matches.isnull().sum())

ID                   0
City                51
Date                 0
Season               0
MatchNumber          0
Team1                0
Team2                0
Venue                0
TossWinner           0
TossDecision         0
SuperOver            4
WinningTeam          4
WonBy                0
Margin              18
method             931
Player_of_Match      4
Team1Players         0
Team2Players         0
Umpire1              0
Umpire2              0
dtype: int64


In [7]:
print(balls.isnull().sum())

ID                       0
innings                  0
overs                    0
ballnumber               0
batter                   0
bowler                   1
non-striker              1
extra_type           55678
batsman_run              1
extras_run               1
total_run                1
non_boundary             1
isWicketDelivery         1
player_out           55846
kind                 55846
fielders_involved    56655
BattingTeam              1
dtype: int64


#Keep Only First Innings
####We want to predict the final score of an innings so we train on:

In [8]:
balls = balls[balls['innings'] == 1]

#Calculate Current Score

In [9]:
balls['current_score'] = balls.groupby('ID')['total_run'].cumsum()

In [10]:
balls[['ID','overs','ballnumber','current_score']].head(20)

,ID,overs,ballnumber,current_score
0,1312200,0,1,0.0
1,1312200,0,2,1.0
2,1312200,0,3,2.0
3,1312200,0,4,2.0
4,1312200,0,5,2.0
5,1312200,0,6,2.0
6,1312200,1,1,2.0
7,1312200,1,2,2.0
8,1312200,1,3,6.0
9,1312200,1,4,6.0


#Calculate Wickets Lost

In [11]:
balls['isWicketDelivery'] = balls['isWicketDelivery'].fillna(0)

balls['wickets'] = balls.groupby('ID')['isWicketDelivery'].cumsum()

In [12]:
balls[['current_score','wickets']].head(20)

,current_score,wickets
0,0.0,0.0
1,1.0,0.0
2,2.0,0.0
3,2.0,0.0
4,2.0,0.0
5,2.0,0.0
6,2.0,0.0
7,2.0,0.0
8,6.0,0.0
9,6.0,0.0


#Create Current Over

In [13]:
balls['current_over'] = balls['overs'] + balls['ballnumber']/10

In [14]:
balls[['ballnumber' , 'current_over']].head(12)

,ballnumber,current_over
0,1,0.1
1,2,0.2
2,3,0.3
3,4,0.4
4,5,0.5
5,6,0.6
6,1,1.1
7,2,1.2
8,3,1.3
9,4,1.4


#Calculate Run Rate

In [15]:
balls['current_rr'] = (balls['current_score'] / balls['current_over'])

In [16]:
balls[['current_over' , 'current_rr']].head(20)

,current_over,current_rr
0,0.1,0.000000
1,0.2,5.000000
2,0.3,6.666667
3,0.4,5.000000
4,0.5,4.000000
5,0.6,3.333333
6,1.1,1.818182
7,1.2,1.666667
8,1.3,4.615385
9,1.4,4.285714


#Final Scores

In [17]:
final_scores = balls.groupby('ID')['current_score'].max().reset_index()
final_scores.rename(columns = {'current_score' : 'final_score'} , inplace = True)

In [18]:
final_scores.head()

,ID,final_score
0,1175366,231.0
1,1175367,175.0
2,1175368,166.0
3,1175369,158.0
4,1175370,170.0


#Merge Final Scores

In [19]:
df = balls.merge(final_scores , on = 'ID')

In [20]:
df.head(10)

,ID,innings,overs,ballnumber,batter,bowler,non-striker,extra_type,batsman_run,extras_run,...,isWicketDelivery,player_out,kind,fielders_involved,BattingTeam,current_score,wickets,current_over,current_rr,final_score
0,1312200,1,0,1,YBK Jaiswal,Mohammed Shami,JC Buttler,NaN,0.0,0.0,...,0.0,NaN,NaN,NaN,Rajasthan Royals,0.0,0.0,0.1,0.000000,130.0
1,1312200,1,0,2,YBK Jaiswal,Mohammed Shami,JC Buttler,legbyes,0.0,1.0,...,0.0,NaN,NaN,NaN,Rajasthan Royals,1.0,0.0,0.2,5.000000,130.0
2,1312200,1,0,3,JC Buttler,Mohammed Shami,YBK Jaiswal,NaN,1.0,0.0,...,0.0,NaN,NaN,NaN,Rajasthan Royals,2.0,0.0,0.3,6.666667,130.0
3,1312200,1,0,4,YBK Jaiswal,Mohammed Shami,JC Buttler,NaN,0.0,0.0,...,0.0,NaN,NaN,NaN,Rajasthan Royals,2.0,0.0,0.4,5.000000,130.0
4,1312200,1,0,5,YBK Jaiswal,Mohammed Shami,JC Buttler,NaN,0.0,0.0,...,0.0,NaN,NaN,NaN,Rajasthan Royals,2.0,0.0,0.5,4.000000,130.0
5,1312200,1,0,6,YBK Jaiswal,Mohammed Shami,JC Buttler,NaN,0.0,0.0,...,0.0,NaN,NaN,NaN,Rajasthan Royals,2.0,0.0,0.6,3.333333,130.0
6,1312200,1,1,1,JC Buttler,Yash Dayal,YBK Jaiswal,NaN,0.0,0.0,...,0.0,NaN,NaN,NaN,Rajasthan Royals,2.0,0.0,1.1,1.818182,130.0
7,1312200,1,1,2,JC Buttler,Yash Dayal,YBK Jaiswal,NaN,0.0,0.0,...,0.0,NaN,NaN,NaN,Rajasthan Royals,2.0,0.0,1.2,1.666667,130.0
8,1312200,1,1,3,JC Buttler,Yash Dayal,YBK Jaiswal,NaN,4.0,0.0,...,0.0,NaN,NaN,NaN,Rajasthan Royals,6.0,0.0,1.3,4.615385,130.0
9,1312200,1,1,4,JC Buttler,Yash Dayal,YBK Jaiswal,NaN,0.0,0.0,...,0.0,NaN,NaN,NaN,Rajasthan Royals,6.0,0.0,1.4,4.285714,130.0


#Venue Information

In [21]:
match_info = matches[['ID','Venue','Team1','Team2']]
df = df.merge(match_info , on = 'ID')

In [22]:
df.head()

,ID,innings,overs,ballnumber,batter,bowler,non-striker,extra_type,batsman_run,extras_run,...,fielders_involved,BattingTeam,current_score,wickets,current_over,current_rr,final_score,Venue,Team1,Team2
0,1312200,1,0,1,YBK Jaiswal,Mohammed Shami,JC Buttler,NaN,0.0,0.0,...,NaN,Rajasthan Royals,0.0,0.0,0.1,0.000000,130.0,"Narendra Modi Stadium, Ahmedabad",Rajasthan Royals,Gujarat Titans
1,1312200,1,0,2,YBK Jaiswal,Mohammed Shami,JC Buttler,legbyes,0.0,1.0,...,NaN,Rajasthan Royals,1.0,0.0,0.2,5.000000,130.0,"Narendra Modi Stadium, Ahmedabad",Rajasthan Royals,Gujarat Titans
2,1312200,1,0,3,JC Buttler,Mohammed Shami,YBK Jaiswal,NaN,1.0,0.0,...,NaN,Rajasthan Royals,2.0,0.0,0.3,6.666667,130.0,"Narendra Modi Stadium, Ahmedabad",Rajasthan Royals,Gujarat Titans
3,1312200,1,0,4,YBK Jaiswal,Mohammed Shami,JC Buttler,NaN,0.0,0.0,...,NaN,Rajasthan Royals,2.0,0.0,0.4,5.000000,130.0,"Narendra Modi Stadium, Ahmedabad",Rajasthan Royals,Gujarat Titans
4,1312200,1,0,5,YBK Jaiswal,Mohammed Shami,JC Buttler,NaN,0.0,0.0,...,NaN,Rajasthan Royals,2.0,0.0,0.5,4.000000,130.0,"Narendra Modi Stadium, Ahmedabad",Rajasthan Royals,Gujarat Titans


#Balls Remaining

In [23]:
balls_played = (df['overs'] * 6 + df['ballnumber'])
df['balls_left'] = 120 - balls_played

In [24]:
df[['current_over' , 'balls_left']].head()

,current_over,balls_left
0,0.1,119
1,0.2,118
2,0.3,117
3,0.4,116
4,0.5,115


# Wickets Left


In [40]:
df['wickets_left'] = 10 - df['wickets']

df[['wickets','wickets_left']].head()

,wickets,wickets_left
0,0.0,10.0
1,0.0,10.0
2,0.0,10.0
3,0.0,10.0
4,0.0,10.0


In [89]:
df['tail_end'] = np.where(
    df['wickets_left'] <= 1,
    1,
    0
)

df[['wickets_left', 'tail_end']].head()

,wickets_left,tail_end
0,10.0,0
1,10.0,0
2,10.0,0
3,10.0,0
4,10.0,0


In [90]:
df['balls_per_wicket'] = (
    df['balls_left'] /
    (df['wickets_left'] + 1)
)

df[['balls_left', 'wickets_left', 'balls_per_wicket']].head()

,balls_left,wickets_left,balls_per_wicket
0,119,10.0,10.818182
1,118,10.0,10.727273
2,117,10.0,10.636364
3,116,10.0,10.545455
4,115,10.0,10.454545


In [91]:
df['death_risk'] = np.where(
    (df['wickets_left'] <= 2) &
    (df['balls_left'] >= 24),
    1,
    0
)

df[['wickets_left', 'balls_left', 'death_risk']].head()

,wickets_left,balls_left,death_risk
0,10.0,119,0
1,10.0,118,0
2,10.0,117,0
3,10.0,116,0
4,10.0,115,0


#Runs in Last 5 Overs

In [44]:
df['last_5_runs'] = (
    df.groupby('ID')['total_run']
      .rolling(30, min_periods=1)
      .sum()
      .reset_index(level=0, drop=True)
)

df[['current_score','last_5_runs']].head(40)

,current_score,last_5_runs
0,0.0,0.0
1,1.0,1.0
2,2.0,2.0
3,2.0,2.0
4,2.0,2.0
5,2.0,2.0
6,2.0,2.0
7,2.0,2.0
8,6.0,6.0
9,6.0,6.0


#Wickets in Last 5 Overs

In [45]:
df['last_5_wickets'] = (
    df.groupby('ID')['isWicketDelivery']
      .rolling(30, min_periods=1)
      .sum()
      .reset_index(level=0, drop=True)
)

df[['wickets','last_5_wickets']].head(40)

,wickets,last_5_wickets
0,0.0,0.0
1,0.0,0.0
2,0.0,0.0
3,0.0,0.0
4,0.0,0.0
5,0.0,0.0
6,0.0,0.0
7,0.0,0.0
8,0.0,0.0
9,0.0,0.0


#Phase

In [51]:
def get_phase(over):

    if over <= 6:
        return 'Powerplay'

    elif over <= 15:
        return 'Middle Overs'

    else:
        return 'Death Overs'

df['phase'] = df['current_over'].apply(get_phase)
df[['current_over','phase']].head(120)

,current_over,phase
0,0.1,Powerplay
1,0.2,Powerplay
2,0.3,Powerplay
3,0.4,Powerplay
4,0.5,Powerplay
...,...,...
115,19.2,Death Overs
116,19.3,Death Overs
117,19.4,Death Overs
118,19.5,Death Overs


#Create Bowling Team

In [52]:
df['BowlingTeam'] = np.where(
    df['BattingTeam'] == df['Team1'],
    df['Team2'],
    df['Team1']
)

df[['BattingTeam','BowlingTeam']].head()

,BattingTeam,BowlingTeam
0,Rajasthan Royals,Gujarat Titans
1,Rajasthan Royals,Gujarat Titans
2,Rajasthan Royals,Gujarat Titans
3,Rajasthan Royals,Gujarat Titans
4,Rajasthan Royals,Gujarat Titans


#Venue Average Score

In [54]:
venue_avg = (
    df.groupby('Venue')['final_score']
      .mean()
)

df['venue_avg'] = df['Venue'].map(venue_avg)
df[['Venue','venue_avg']].head()

,Venue,venue_avg
0,"Narendra Modi Stadium, Ahmedabad",154.296383
1,"Narendra Modi Stadium, Ahmedabad",154.296383
2,"Narendra Modi Stadium, Ahmedabad",154.296383
3,"Narendra Modi Stadium, Ahmedabad",154.296383
4,"Narendra Modi Stadium, Ahmedabad",154.296383


#Runs Left

In [25]:
df['runs_left'] = (df['final_score'] - df['current_score'])

In [26]:
df[['current_score' , 'runs_left']].head()

,current_score,runs_left
0,0.0,130.0
1,1.0,129.0
2,2.0,128.0
3,2.0,128.0
4,2.0,128.0


#Remove Invalid Rows

In [27]:
df = df[df['balls_left'] >= 0]
df = df[df['runs_left'] >= 0]

#Select Features

In [92]:
final_df = df[[
    'Venue',
    'BattingTeam',
    'BowlingTeam',
    'current_over',
    'current_score',
    'extras_run',
    'wickets_left',
    'current_rr',
    'balls_left',
    'last_5_runs',
    'last_5_wickets',
    'phase',
    'venue_avg',
    'tail_end',
    'balls_per_wicket',
    'death_risk',
    'final_score'
]]

In [93]:
final_df.head()

,Venue,BattingTeam,BowlingTeam,current_over,current_score,extras_run,wickets_left,current_rr,balls_left,last_5_runs,last_5_wickets,phase,venue_avg,tail_end,balls_per_wicket,death_risk,final_score
0,"Narendra Modi Stadium, Ahmedabad",Rajasthan Royals,Gujarat Titans,0.1,0.0,0.0,10.0,0.000000,119,0.0,0.0,Powerplay,154.296383,0,10.818182,0,130.0
1,"Narendra Modi Stadium, Ahmedabad",Rajasthan Royals,Gujarat Titans,0.2,1.0,1.0,10.0,5.000000,118,1.0,0.0,Powerplay,154.296383,0,10.727273,0,130.0
2,"Narendra Modi Stadium, Ahmedabad",Rajasthan Royals,Gujarat Titans,0.3,2.0,0.0,10.0,6.666667,117,2.0,0.0,Powerplay,154.296383,0,10.636364,0,130.0
3,"Narendra Modi Stadium, Ahmedabad",Rajasthan Royals,Gujarat Titans,0.4,2.0,0.0,10.0,5.000000,116,2.0,0.0,Powerplay,154.296383,0,10.545455,0,130.0
4,"Narendra Modi Stadium, Ahmedabad",Rajasthan Royals,Gujarat Titans,0.5,2.0,0.0,10.0,4.000000,115,2.0,0.0,Powerplay,154.296383,0,10.454545,0,130.0


#Remove Initial Overs

In [94]:
final_df = final_df[final_df['current_over'] >= 6]

In [31]:
final_df.head()

,Venue,BattingTeam,current_over,current_score,extras_run,wickets,current_rr,balls_left,final_score
36,"Narendra Modi Stadium, Ahmedabad",Rajasthan Royals,6.1,44.0,0.0,1.0,7.213115,83,130.0
37,"Narendra Modi Stadium, Ahmedabad",Rajasthan Royals,6.2,45.0,0.0,1.0,7.258065,82,130.0
38,"Narendra Modi Stadium, Ahmedabad",Rajasthan Royals,6.3,45.0,0.0,1.0,7.142857,81,130.0
39,"Narendra Modi Stadium, Ahmedabad",Rajasthan Royals,6.4,49.0,0.0,1.0,7.656250,80,130.0
40,"Narendra Modi Stadium, Ahmedabad",Rajasthan Royals,6.5,53.0,0.0,1.0,8.153846,79,130.0


#Encode Categorical Variables

In [73]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

#Define X and y

In [95]:
X = final_df.drop('final_score', axis=1)
y = final_df['final_score']

#Train Test Split

In [96]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

#Preprocessing Pipeline

In [97]:
trf = ColumnTransformer(
[
    (
        'ohe',
        OneHotEncoder(handle_unknown='ignore'),
        [
            'Venue',
            'BattingTeam',
            'BowlingTeam',
            'phase'
        ]
    )
],
remainder='passthrough'
)

#Random Forest

In [98]:
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor

pipe_rf = Pipeline(
[
    ('transformer', trf),

    ('model',
     RandomForestRegressor(
         n_estimators=300,
         max_depth=20,
         min_samples_leaf=3,
         random_state=42,
         n_jobs=-1
     ))
]
)

pipe_rf.fit(X_train, y_train)

pred_rf = pipe_rf.predict(X_test)

In [99]:
pipe_rf.fit(X_train, y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/compose/_column_transformer.py:1667: FutureWarning: 
The format of the columns of the 'remainder' transformer in ColumnTransformer.transformers_ will change in version 1.7 to match the format of the other transformers.
At the moment the remainder columns are stored as indices (of type int). With the same ColumnTransformer configuration, in the future they will be stored as column names (of type str).
To use the new behavior now and suppress this warning, use ColumnTransformer(force_int_remainder_cols=False).

  warnings.warn(


Pipeline(steps=[('transformer',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('ohe',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['Venue', 'BattingTeam',
                                                   'BowlingTeam', 'phase'])])),
                ('model',
                 RandomForestRegressor(max_depth=20, min_samples_leaf=3,
                                       n_estimators=300, n_jobs=-1,
                                       random_state=42))])

#Prediction

In [100]:
pred_rf = pipe_rf.predict(X_test)

#Evaluation

In [101]:
from sklearn.metrics import *

print("MAE :", mean_absolute_error(y_test, pred_rf))
print("RMSE :", np.sqrt(mean_squared_error(y_test, pred_rf)))
print("R2 Score :", r2_score(y_test, pred_rf))

MAE : 2.4160184806110263
RMSE : 4.935352918474275
R2 Score : 0.9706867305161339


#Save Model

In [102]:
import joblib

joblib.dump(pipe_rf , "cricket_model.pkl")

['cricket_model.pkl']

#Predicting Live Score

In [109]:
# ============================
# 🏏 IPL LIVE SCORE PREDICTOR
# ============================

import pandas as pd
import numpy as np

print("="*65)
print("🏏             IPL LIVE SCORE PREDICTOR             🏏")
print("="*65)

# USER INPUTS

venue = input("📍 Enter Venue                     : ")
batting_team = input("🏏 Enter Batting Team              : ")
bowling_team = input("🎯 Enter Bowling Team              : ")
current_over = float(input("⏱️ Enter Overs Played              : "))
current_score = int(input("🔢 Enter Current Score             : "))
extras_run = int(input("➕ Enter Extras Runs               : "))
wickets = int(input("❌ Enter Wickets Lost              : "))
last_5_runs = int(input("🔥 Runs in Last 5 Overs            : "))
last_5_wickets = int(input("💥 Wickets Lost in Last 5 Overs    : "))

# CALCULATIONS

whole_over = int(current_over)
balls = round((current_over - whole_over) * 10)

balls_played = whole_over * 6 + balls
balls_left = 120 - balls_played

wickets_left = 10 - wickets

if current_over > 0:
    current_rr = current_score / current_over
else:
    current_rr = 0

# Innings Phase

if current_over <= 6:
    phase = "Powerplay"
elif current_over <= 15:
    phase = "Middle"
else:
    phase = "Death"

# Venue Average

venue_avg = final_df[
    final_df['Venue'] == venue
]['final_score'].mean()

if np.isnan(venue_avg):
    venue_avg = final_df['final_score'].mean()

# Tail-End Situation

if wickets_left <= 1:
    tail_end = 1
else:
    tail_end = 0

# Balls Per Wicket

balls_per_wicket = balls_left / (wickets_left + 1)

# Death Risk

if wickets_left <= 2 and balls_left >= 24:
    death_risk = 1
else:
    death_risk = 0

# VALID INPUTS

if (
    0 <= wickets < 10 and
    0 < current_over <= 20 and
    extras_run >= 0 and
    current_score >= 0 and
    balls_left >= 0 and
    0 <= last_5_wickets <= 5 and
    last_5_runs >= 0
):

    sample = pd.DataFrame({
        'Venue': [venue],
        'BattingTeam': [batting_team],
        'BowlingTeam': [bowling_team],
        'current_over': [current_over],
        'current_score': [current_score],
        'extras_run': [extras_run],
        'wickets_left': [wickets_left],
        'current_rr': [current_rr],
        'balls_left': [balls_left],
        'last_5_runs': [last_5_runs],
        'last_5_wickets': [last_5_wickets],
        'phase': [phase],
        'venue_avg': [venue_avg],
        'tail_end': [tail_end],
        'balls_per_wicket': [balls_per_wicket],
        'death_risk': [death_risk]
    })

    prediction = pipe_rf.predict(sample)

    pred = int(prediction[0])

    # Cricket Logic Corrections

    if wickets_left == 1:
        pred = min(pred, current_score + 20)

    elif wickets_left == 2:
        pred = min(pred, current_score + 35)

    pred = max(pred, current_score)

    lower = pred - 5
    upper = pred + 5

    # MATCH SUMMARY

    print("\n" + "="*65)
    print("📊                 MATCH SUMMARY")
    print("="*65)

    print(f"🏟️ Venue                      : {venue}")
    print(f"🏏 Batting Team               : {batting_team}")
    print(f"🎯 Bowling Team               : {bowling_team}")
    print(f"🔢 Current Score              : {current_score}/{wickets}")
    print(f"⏱️ Overs Played               : {current_over}")
    print(f"📈 Current Run Rate           : {current_rr:.2f}")
    print(f"🎯 Balls Left                 : {balls_left}")
    print(f"🧤 Wickets Left               : {wickets_left}")
    print(f"🔥 Runs in Last 5 Overs       : {last_5_runs}")
    print(f"💥 Wickets in Last 5 Overs    : {last_5_wickets}")
    print(f"⚡ Innings Phase              : {phase}")
    print(f"🏟️ Venue Average Score        : {venue_avg:.0f}")

    print("="*65)
    print("🧠              ADVANCED ANALYTICS")
    print("="*65)

    print(f"🏁 Tail-End Situation         : {'Yes' if tail_end == 1 else 'No'}")
    print(f"⚠️ Death Risk                 : {'High' if death_risk == 1 else 'Normal'}")
    print(f"📊 Balls per Wicket           : {balls_per_wicket:.2f}")

    print("="*65)
    print(f"🤖 Predicted Final Score      : {pred} Runs")
    print(f"📌 Predicted Score Range      : {lower} - {upper} Runs")
    print("="*65)
    print("✅ Prediction Generated Successfully!")
    print("="*65)

    if pred >= 200:
        status = "🔥 Explosive Innings Expected"
    elif pred >= 170:
        status = "💪 Competitive Total Expected"
    elif pred >= 150:
        status = "⚡ Average Total Expected"
    else:
        status = "💔 Below Par Score Expected"

    print(f"🏆 Match Insight              : {status}")
    print("="*65)

# ALL OUT CASE

elif (
    wickets == 10 and
    0 < current_over <= 20 and
    extras_run >= 0
):

    print("\n" + "="*65)
    print("🏏                 INNINGS OVER")
    print("="*65)

    print(f"🏏 Team                       : {batting_team}")
    print(f"🎯 Against                    : {bowling_team}")
    print(f"🔢 Final Score                : {current_score} ALL OUT")
    print(f"⏱️ Overs Played               : {current_over}")

    print("="*65)

    if current_score >= 200:
        status = "🔥 Explosive Innings"
    elif current_score >= 170:
        status = "💪 Competitive Total"
    elif current_score >= 150:
        status = "⚡ Average Total"
    else:
        status = "💔 Below Par Score"

    print(f"🏆 Match Insight              : {status}")
    print("="*65)

# INVALID INFORMATION

else:

    print("\n" + "="*65)
    print("❌            INVALID MATCH INPUT REPORT")
    print("="*65)

    if wickets < 0 or wickets > 10:
        print(f"🚫 Invalid Wickets Entered       : {wickets}")
        print("💡 Wickets must be between 0 and 10.\n")

    if current_over <= 0 or current_over > 20:
        print(f"🚫 Invalid Overs Entered         : {current_over}")
        print("💡 Overs must be between 0.1 and 20.0.\n")

    if extras_run < 0:
        print(f"🚫 Invalid Extras Entered        : {extras_run}")
        print("💡 Extras cannot be negative.\n")

    if current_score < 0:
        print(f"🚫 Invalid Score Entered         : {current_score}")
        print("💡 Score cannot be negative.\n")

    if last_5_runs < 0:
        print(f"🚫 Invalid Last 5 Overs Runs     : {last_5_runs}")
        print("💡 Runs cannot be negative.\n")

    if last_5_wickets < 0 or last_5_wickets > 5:
        print(f"🚫 Invalid Last 5 Wickets        : {last_5_wickets}")
        print("💡 Wickets must be between 0 and 5.\n")

    print("="*65)
    print("⚠️ Please enter valid cricket match information.")
    print("🔄 Restart the predictor and try again.")
    print("="*65)

🏏             IPL LIVE SCORE PREDICTOR             🏏
📍 Enter Venue                     : Narendra Modi Stadium
🏏 Enter Batting Team              : Gujarat Titans
🎯 Enter Bowling Team              : Royal Challengers Bangalore
⏱️ Enter Overs Played              : 12.1
🔢 Enter Current Score             : 73
➕ Enter Extras Runs               : 2
❌ Enter Wickets Lost              : 4
🔥 Runs in Last 5 Overs            : 51
💥 Wickets Lost in Last 5 Overs    : 2

📊                 MATCH SUMMARY
🏟️ Venue                      : Narendra Modi Stadium
🏏 Batting Team               : Gujarat Titans
🎯 Bowling Team               : Royal Challengers Bangalore
🔢 Current Score              : 73/4
⏱️ Overs Played               : 12.1
📈 Current Run Rate           : 6.03
🎯 Balls Left                 : 47
🧤 Wickets Left               : 6
🔥 Runs in Last 5 Overs       : 51
💥 Wickets in Last 5 Overs    : 2
⚡ Innings Phase              : Middle
🏟️ Venue Average Score        : 167
🧠              ADVANCED ANALYTI